# 07 – Phân cụm: K-Means, Agglomerative, DBSCAN/HDBSCAN, t-SNE

**Người thực hiện:** thinh (Nguyễn Vĩnh Thịnh)  
**Mục tiêu:** Trả lời RQ3: phân cụm không giám sát tái tạo cấu trúc ngành hàng tới đâu?

**Quy tắc:** chạy TỪNG ô một, đổi tham số rồi chạy lại; mỗi ngày làm xong thì `git add` + `git commit -m "<mô tả đúng việc đã làm>"` + `git push`. Cuối buổi xuất HTML: `python ../scripts/export_notebook_html.py <notebook> thinh`.

**Các bước:**
1. K-Means: quét K (6–20) và số chiều SVD.
2. Agglomerative (mẫu 5.000–8.000 điểm): linkage ward/average/complete.
3. DBSCAN: quét eps, min_samples; HDBSCAN: min_cluster_size.
4. t-SNE trên mẫu 5.000 điểm, tô màu theo ngành và theo cụm.
5. Chọn thuật toán tốt nhất, giải thích Silhouette/ARI/NMI bằng lời.

In [1]:
import sys, json, pandas as pd, numpy as np, matplotlib.pyplot as plt
sys.path.append('../src')
from run_experiment import run
from figlog import save_fig
from exp_logger import log_generic
from common import load_data, clean_title


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, HDBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

df = load_data(); df['clean'] = df.title.map(clean_title); y = df.category_label.values
Xt = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, token_pattern=r"(?u)\b[\w\-\.\+]+\b").fit_transform(df.clean)
def embed(m):
    return Normalizer().fit_transform(TruncatedSVD(m, random_state=42).fit_transform(Xt))

def score(Z, lab, algo, **cfg):
    rng = np.random.RandomState(42); s = rng.choice(len(Z), min(10000, len(Z)), replace=False)
    ok = lab[s] != -1  # bỏ điểm nhiễu của DBSCAN khi tính silhouette
    r = dict(algo=algo, **cfg, n_clusters=len(set(lab)) - (1 if -1 in lab else 0), noise_pct=float((lab == -1).mean()),
             silhouette=silhouette_score(Z[s][ok], lab[s][ok], metric='cosine') if len(set(lab[s][ok])) > 1 else float('nan'),
             ari=adjusted_rand_score(yy, lab), nmi=normalized_mutual_info_score(yy, lab))
    log_generic('thinh', 'cluster_log', r); print(r); return r

## 1. K-Means

In [5]:
res = []
for k in range(6, 21, 2):
    lab = KMeans(k, n_init=10, random_state=42).fit_predict(Z)
    res.append(score(Z, lab, 'kmeans', K=k, svd=50))
ket_qua = pd.DataFrame(res)[['K', 'silhouette', 'ari', 'nmi']]
ket_qua

{'algo': 'kmeans', 'K': 6, 'svd': 50, 'n_clusters': 6, 'noise_pct': 0.0, 'silhouette': 0.21560822546957853, 'ari': 0.38795468553117646, 'nmi': 0.6171020800332649}
{'algo': 'kmeans', 'K': 8, 'svd': 50, 'n_clusters': 8, 'noise_pct': 0.0, 'silhouette': 0.26888575323135444, 'ari': 0.4831093153719097, 'nmi': 0.7167283195830451}
{'algo': 'kmeans', 'K': 10, 'svd': 50, 'n_clusters': 10, 'noise_pct': 0.0, 'silhouette': 0.3016820669038938, 'ari': 0.6229551874853975, 'nmi': 0.7318251931951658}
{'algo': 'kmeans', 'K': 12, 'svd': 50, 'n_clusters': 12, 'noise_pct': 0.0, 'silhouette': 0.30791268597888705, 'ari': 0.567926709838571, 'nmi': 0.7111196133879203}
{'algo': 'kmeans', 'K': 14, 'svd': 50, 'n_clusters': 14, 'noise_pct': 0.0, 'silhouette': 0.31343436497090404, 'ari': 0.5444062090302868, 'nmi': 0.7011448477035328}
{'algo': 'kmeans', 'K': 16, 'svd': 50, 'n_clusters': 16, 'noise_pct': 0.0, 'silhouette': 0.3377260258364998, 'ari': 0.5459624658142548, 'nmi': 0.7040620681187659}
{'algo': 'kmeans', 'K'

,K,silhouette,ari,nmi
0,6,0.215608,0.387955,0.617102
1,8,0.268886,0.483109,0.716728
2,10,0.301682,0.622955,0.731825
3,12,0.307913,0.567927,0.711120
4,14,0.313434,0.544406,0.701145
5,16,0.337726,0.545962,0.704062
6,18,0.344808,0.551461,0.695845
7,20,0.347170,0.463338,0.660763


 Kết luận
- **Số cụm K tốt nhất:** Dựa trên quá trình quét K từ 6 đến 20 với SVD-50, **K = 10** là giá trị tối ưu nhất vì đạt điểm NMI cao nhất (`0.7318`), đồng thời chỉ số ARI cũng đạt mức cao nhất (`0.623`).
- **Đánh giá NMI sau khi sửa TF-IDF:** Sau khi bổ sung đúng `token_pattern` cho `TfidfVectorizer`, chỉ số NMI tại K=10 đã tăng lên và phục hồi chính xác bằng mức `0.7318` của bản giữa kỳ (đúng như kỳ vọng đặt ra).
- **So sánh giữa Silhouette và NMI/ARI:** 
  - Chỉ số **Silhouette** tăng dần theo số cụm lớn (đạt cao nhất ở $K=20$ với giá trị `0.347`), do đặc thù không gian vector văn bản thưa thớt khiến Silhouette có xu hướng ưa chuộng việc phân chia thành nhiều cụm nhỏ hơn.
  - Ngược lại, **NMI** và **ARI** (các chỉ số đánh giá có nhãn ground-truth) đạt đỉnh rõ rệt tại $K=10$ rồi giảm dần. Do đó, $K=10$ là lựa chọn chính xác nhất về mặt ngữ nghĩa phân cụm thực tế so với nhãn gốc của tập dữ liệu.

### 1b. So sánh số chiều SVD (50/100/150) và MiniBatch K-Means, K = 10

In [24]:
from sklearn.cluster import MiniBatchKMeans

ket_qua_svd = []
for m in [50, 100, 150]:
    Z = embed(m); yy = y
    # K-Means đầy đủ
    lab = KMeans(10, n_init=10, random_state=42).fit_predict(Z)
    ket_qua_svd.append(score(Z, lab, 'kmeans', K=10, svd=m))
    # MiniBatch K-Means: nhanh hơn, cập nhật tâm cụm theo từng lô 2048 điểm
    lab = MiniBatchKMeans(10, n_init=10, batch_size=2048, random_state=42).fit_predict(Z)
    ket_qua_svd.append(score(Z, lab, 'minibatch_kmeans', K=10, svd=m))

bang_svd = pd.DataFrame(ket_qua_svd)[['algo', 'svd', 'silhouette', 'ari', 'nmi']]
bang_svd

{'algo': 'kmeans', 'K': 10, 'svd': 50, 'n_clusters': 10, 'noise_pct': 0.0, 'silhouette': 0.3016820669038938, 'ari': 0.6229551874853975, 'nmi': 0.7318251931951658}
{'algo': 'minibatch_kmeans', 'K': 10, 'svd': 50, 'n_clusters': 10, 'noise_pct': 0.0, 'silhouette': 0.2685280400090133, 'ari': 0.5395343446172596, 'nmi': 0.6708651425765852}
{'algo': 'kmeans', 'K': 10, 'svd': 100, 'n_clusters': 10, 'noise_pct': 0.0, 'silhouette': 0.19342657740185434, 'ari': 0.5033999276095351, 'nmi': 0.6915784700914626}
{'algo': 'minibatch_kmeans', 'K': 10, 'svd': 100, 'n_clusters': 10, 'noise_pct': 0.0, 'silhouette': 0.17964320509773504, 'ari': 0.5449516391783692, 'nmi': 0.7056940362202965}
{'algo': 'kmeans', 'K': 10, 'svd': 150, 'n_clusters': 10, 'noise_pct': 0.0, 'silhouette': 0.15458698244485894, 'ari': 0.4292817830724501, 'nmi': 0.669244583346262}
{'algo': 'minibatch_kmeans', 'K': 10, 'svd': 150, 'n_clusters': 10, 'noise_pct': 0.0, 'silhouette': 0.15350023027183524, 'ari': 0.5025896103080679, 'nmi': 0.653

,algo,svd,silhouette,ari,nmi
0,kmeans,50,0.301682,0.622955,0.731825
1,minibatch_kmeans,50,0.268528,0.539534,0.670865
2,kmeans,100,0.193427,0.503400,0.691578
3,minibatch_kmeans,100,0.179643,0.544952,0.705694
4,kmeans,150,0.154587,0.429282,0.669245
5,minibatch_kmeans,150,0.153500,0.502590,0.653174


In [25]:
fig, ax = plt.subplots(figsize=(7, 4))
for algo, d in bang_svd.groupby('algo'):
    ax.plot(d.svd, d.nmi, 'o-', label=f'{algo} – NMI')
    ax.plot(d.svd, d.silhouette, 's--', label=f'{algo} – Silhouette')
ax.set(xlabel='Số chiều SVD', ylabel='Giá trị', title='K = 10: chỉ số theo số chiều SVD')
ax.legend(fontsize=8)
save_fig(fig, 'thinh_so_sanh_svd', notebook='07_Thinh_PhanCum.ipynb', member='thinh',
         cell='Mục 1b - so sánh SVD', y_nghia='So sánh các chỉ số NMI và Silhouette theo các mức số chiều SVD 50, 100, 150')

Đã lưu thinh_so_sanh_svd.png (ghi nguồn vào figures/FIGURE_INDEX.csv)


WindowsPath('D:/mayhoc-git/DoAnCuoiKy/figures/thinh_so_sanh_svd.png')

> **Nhật ký lần chạy (14/11):**
> - Kết quả chính: SVD 50 cho kết quả NMI tốt nhất (0.7318), số chiều càng tăng thì NMI có xu hướng giảm.
> - Nhận xét: Số chiều SVD 50 giữ lại các đặc trưng quan trọng nhất; tăng chiều lên 150 có thể mang lại nhiều nhiễu hơn làm giảm độ chính xác phân cụm. MiniBatch K-Means chạy nhanh hơn đáng kể nhưng cho điểm số tương đương.
> - Lần sau thử: Chuyển sang thuật toán Agglomerative Clustering.

In [3]:
Z = embed(50); yy = y
lab = KMeans(10, n_init=10, random_state=42).fit_predict(Z)
score(Z, lab, 'kmeans', K=10, svd=50)

{'algo': 'kmeans', 'K': 10, 'svd': 50, 'n_clusters': 10, 'noise_pct': 0.0, 'silhouette': 0.3016820669038938, 'ari': 0.6229551874853975, 'nmi': 0.7318251931951658}


{'algo': 'kmeans',
 'K': 10,
 'svd': 50,
 'n_clusters': 10,
 'noise_pct': 0.0,
 'silhouette': 0.3016820669038938,
 'ari': 0.6229551874853975,
 'nmi': 0.7318251931951658}


> - Kết quả chính:0.2936
> - Nhận xét:0.5636
> - Lần sau thử:k=8,K=12

## 2. Agglomerative (trên mẫu)

In [26]:
# Agglomerative tính khoảng cách mọi cặp điểm (bộ nhớ ~ n^2), nên chạy trên mẫu 6.000 điểm
idx = np.random.RandomState(0).choice(len(df), 6000, replace=False)
Zs, yy = embed(50)[idx], y[idx]

ket_qua_agg = []
for lk in ['ward', 'average', 'complete']:
    lab = AgglomerativeClustering(10, linkage=lk).fit_predict(Zs)
    ket_qua_agg.append(score(Zs, lab, 'agglomerative', K=10, linkage=lk, svd=50, n_sample=6000))

bang_agg = pd.DataFrame(ket_qua_agg)[['linkage', 'silhouette', 'ari', 'nmi']]
bang_agg

{'algo': 'agglomerative', 'K': 10, 'linkage': 'ward', 'svd': 50, 'n_sample': 6000, 'n_clusters': 10, 'noise_pct': 0.0, 'silhouette': 0.2557383139748478, 'ari': 0.4060009144442757, 'nmi': 0.6125619116175389}
{'algo': 'agglomerative', 'K': 10, 'linkage': 'average', 'svd': 50, 'n_sample': 6000, 'n_clusters': 10, 'noise_pct': 0.0, 'silhouette': 0.2570825229116946, 'ari': 0.5543065323359407, 'nmi': 0.6532243389380232}
{'algo': 'agglomerative', 'K': 10, 'linkage': 'complete', 'svd': 50, 'n_sample': 6000, 'n_clusters': 10, 'noise_pct': 0.0, 'silhouette': 0.05900134601417009, 'ari': 0.13323040631789038, 'nmi': 0.24783068188289928}


,linkage,silhouette,ari,nmi
0,ward,0.255738,0.406001,0.612562
1,average,0.257083,0.554307,0.653224
2,complete,0.059001,0.133230,0.247831


> **Nhật ký lần chạy (14/11):**
> - Kết quả chính: Kiểu liên kết `average` cho NMI tốt nhất (~0.6532), tiếp theo là `ward` (~0.6126), còn `complete` rất kém (~0.2478).
> - Nhận xét: Kiểu `complete` (liên kết hoàn toàn) không phù hợp với phân bố dữ liệu văn bản này do dễ bị ảnh hưởng bởi điểm ngoại lai. So với K-Means (SVD 50 với NMI 0.7318), K-Means vẫn vượt trội hơn Agglomerative Clustering về mặt tối ưu hóa cụm.
> - Lần sau thử: DBSCAN/HDBSCAN (mục 3)

In [ ]:
idx = np.random.RandomState(0).choice(len(df), 6000, replace=False)
Zs, yy = embed(50)[idx], y[idx]
lab = AgglomerativeClustering(10, linkage='ward').fit_predict(Zs)
score(Zs, lab, 'agglomerative', K=10, linkage='ward', n_sample=6000)

> **Nhật ký lần chạy** – ngày giờ đã tự ghi trong `experiments/thinh/`. Sau khi xem kết quả, mở file CSV và **tự viết** cột `nhan_xet` (thấy gì?) và `rut_ra` (quyết định gì cho lần sau?). Viết lại ngắn gọn ở ô này:
>
> - Kết quả chính: …
> - Nhận xét: …
> - Lần sau thử: …

## 3. DBSCAN / HDBSCAN

In [4]:
from sklearn.neighbors import NearestNeighbors

# Dùng lại đúng mẫu 6.000 điểm như mục 2 (không chạy lại ô Agglomerative)
idx = np.random.RandomState(0).choice(len(df), 6000, replace=False)
Zs, yy = embed(50)[idx], y[idx]

# k-distance: khoảng cách từ mỗi điểm tới láng giềng thứ 10, sắp xếp tăng dần.
d, _ = NearestNeighbors(n_neighbors=10).fit(Zs).kneighbors(Zs)
kd = np.sort(d[:, -1])
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(kd)
for e in [0.3, 0.4, 0.5, 0.6]:
    ax.axhline(e, ls='--', lw=0.8, color='gray')
ax.set(xlabel='Các điểm, sắp xếp theo khoảng cách tăng dần', ylabel='Khoảng cách tới láng giềng thứ 10',
       title='Biểu đồ k-distance (k = 10) để chọn eps cho DBSCAN')
save_fig(fig, 'thinh_k_distance', notebook='07_Thinh_PhanCum.ipynb', member='thinh', cell='Mục 3 - k-distance',
         y_nghia='Khoảng cách tới láng giềng thứ 10 tăng đều, không có khuỷu rõ (trung vị 0.39, 90% điểm dưới 0.65), nên quét eps trong khoảng 0.3 đến 0.6.')
print('Trung vị khoảng cách:', round(float(np.median(kd)), 3),
      ' | 90% điểm có khoảng cách dưới:', round(float(np.percentile(kd, 90)), 3))


Đã lưu thinh_k_distance.png (ghi nguồn vào figures/FIGURE_INDEX.csv)
Trung vị khoảng cách: 0.39  | 90% điểm có khoảng cách dưới: 0.653


In [6]:
ket_qua_db = []
# Quét eps với min_samples = 10
for eps in [0.3, 0.4, 0.5, 0.6]:
    lab = DBSCAN(eps=eps, min_samples=10).fit_predict(Zs)
    ket_qua_db.append(score(Zs, lab, 'dbscan', eps=eps, min_samples=10, svd=50, n_sample=6000))
# Giữ eps = 0.5, đổi min_samples
for ms in [5, 20]:
    lab = DBSCAN(eps=0.5, min_samples=ms).fit_predict(Zs)
    ket_qua_db.append(score(Zs, lab, 'dbscan', eps=0.5, min_samples=ms, svd=50, n_sample=6000))

bang_db = pd.DataFrame(ket_qua_db)[['eps', 'min_samples', 'n_clusters', 'noise_pct', 'silhouette', 'ari', 'nmi']]
bang_db


{'algo': 'dbscan', 'eps': 0.3, 'min_samples': 10, 'svd': 50, 'n_sample': 6000, 'n_clusters': 65, 'noise_pct': 0.5643333333333334, 'silhouette': 0.6784265729898814, 'ari': 0.05366673230858713, 'nmi': 0.39381553059999275}
{'algo': 'dbscan', 'eps': 0.4, 'min_samples': 10, 'svd': 50, 'n_sample': 6000, 'n_clusters': 59, 'noise_pct': 0.36283333333333334, 'silhouette': 0.3206902998000919, 'ari': 0.2392686364774728, 'nmi': 0.4992071938387587}
{'algo': 'dbscan', 'eps': 0.5, 'min_samples': 10, 'svd': 50, 'n_sample': 6000, 'n_clusters': 40, 'noise_pct': 0.19933333333333333, 'silhouette': 0.24481120066301607, 'ari': 0.33079832369690665, 'nmi': 0.533391187776899}
{'algo': 'dbscan', 'eps': 0.6, 'min_samples': 10, 'svd': 50, 'n_sample': 6000, 'n_clusters': 7, 'noise_pct': 0.06216666666666667, 'silhouette': 0.0002709203176403277, 'ari': 0.013250826365358005, 'nmi': 0.12186958668114144}
{'algo': 'dbscan', 'eps': 0.5, 'min_samples': 5, 'svd': 50, 'n_sample': 6000, 'n_clusters': 53, 'noise_pct': 0.111833

,eps,min_samples,n_clusters,noise_pct,silhouette,ari,nmi
0,0.3,10,65,0.564333,0.678427,0.053667,0.393816
1,0.4,10,59,0.362833,0.320690,0.239269,0.499207
2,0.5,10,40,0.199333,0.244811,0.330798,0.533391
3,0.6,10,7,0.062167,0.000271,0.013251,0.121870
4,0.5,5,53,0.111833,0.077143,0.393648,0.554132
5,0.5,20,27,0.299000,0.380030,0.285341,0.523278


> **Nhật ký lần chạy (15/11) – DBSCAN:**
> - Kết quả chính: Quét eps từ 0.3 đến 0.6 và thay đổi min_samples tại 0.5.
> - Nhận xét: eps nhỏ (0.3) tạo ra quá nhiều cụm và đẩy hơn nửa dữ liệu vào nhiễu (56.4%), khiến phần còn lại có Silhouette ảo rất cao (0.678) nhưng NMI rất thấp. Khi tăng eps lên 0.6, mô hình gom hầu hết thành 1 cụm lớn. Cấu hình eps=0.5, min_samples=5 cho NMI tốt nhất nhóm (0.554).
> - Lần sau thử: Thử nghiệm với HDBSCAN để xem xét phân cụm dựa trên mật độ phân cấp.
```

In [7]:
ket_qua_hdb = []
for m in [30, 50, 100, 200]:
    lab = HDBSCAN(min_cluster_size=m).fit_predict(Zs)
    ket_qua_hdb.append(score(Zs, lab, 'hdbscan', min_cluster_size=m, svd=50, n_sample=6000))

bang_hdb = pd.DataFrame(ket_qua_hdb)[['min_cluster_size', 'n_clusters', 'noise_pct', 'silhouette', 'ari', 'nmi']]
bang_hdb


d:\mayhoc-git\DoAnCuoiKy\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


{'algo': 'hdbscan', 'min_cluster_size': 30, 'svd': 50, 'n_sample': 6000, 'n_clusters': 32, 'noise_pct': 0.49366666666666664, 'silhouette': 0.6563313103247153, 'ari': 0.08691765420681562, 'nmi': 0.4300456553799975}


d:\mayhoc-git\DoAnCuoiKy\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


{'algo': 'hdbscan', 'min_cluster_size': 50, 'svd': 50, 'n_sample': 6000, 'n_clusters': 18, 'noise_pct': 0.5103333333333333, 'silhouette': 0.5886192928296335, 'ari': 0.16048276537244036, 'nmi': 0.4930415809782682}


d:\mayhoc-git\DoAnCuoiKy\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


{'algo': 'hdbscan', 'min_cluster_size': 100, 'svd': 50, 'n_sample': 6000, 'n_clusters': 8, 'noise_pct': 0.5055, 'silhouette': 0.49247405593705523, 'ari': 0.19454913143196206, 'nmi': 0.5325327744169945}


d:\mayhoc-git\DoAnCuoiKy\.venv\Lib\site-packages\sklearn\cluster\_hdbscan\hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


{'algo': 'hdbscan', 'min_cluster_size': 200, 'svd': 50, 'n_sample': 6000, 'n_clusters': 3, 'noise_pct': 0.6566666666666666, 'silhouette': 0.30468364338098847, 'ari': 0.06905252829045357, 'nmi': 0.2869678982912347}


,min_cluster_size,n_clusters,noise_pct,silhouette,ari,nmi
0,30,32,0.493667,0.656331,0.086918,0.430046
1,50,18,0.510333,0.588619,0.160483,0.493042
2,100,8,0.505500,0.492474,0.194549,0.532533
3,200,3,0.656667,0.304684,0.069053,0.286968


> **Nhật ký lần chạy (15/11) – HDBSCAN:**
> - Kết quả chính: Quét min_cluster_size từ 30 đến 200.
> - Nhận xét: min_cluster_size = 100 cho NMI tốt nhất (0.533) với 8 cụm. Tỷ lệ nhiễu của HDBSCAN luôn ở mức cao (khoảng 50% trở lên), cho thấy dữ liệu văn bản sau SVD có mật độ phân bố không đồng đều rõ rệt giữa các cụm.
> - Lần sau thử: Tổng hợp so sánh toàn bộ các thuật toán.


In [8]:
log = pd.read_csv('../experiments/thinh/cluster_log.csv')
best = log.sort_values('nmi', ascending=False).groupby('algo').head(1).set_index('algo').sort_values('nmi')

fig, ax = plt.subplots(figsize=(8, 4))
best[['nmi', 'ari']].plot.barh(ax=ax)
for i, v in enumerate(best.nmi):
    ax.text(v + 0.01, i - 0.1, f'{v:.3f}', fontsize=8)
ax.set(xlim=(0, 0.85), xlabel='Giá trị (càng cao càng khớp với ngành thật)', ylabel='',
       title='Cấu hình tốt nhất của mỗi thuật toán phân cụm (xếp theo NMI)')
save_fig(fig, 'thinh_so_sanh_thuat_toan', notebook='07_Thinh_PhanCum.ipynb', member='thinh', cell='Mục 3 - so sánh',
         y_nghia='K-Means SVD 50 có NMI cao nhất (0.732), tiếp theo MiniBatch K-Means và Agglomerative average; DBSCAN và HDBSCAN thấp nhất vì gán nhiều điểm là nhiễu và tạo số cụm khác xa 10 ngành.')
best[['K', 'svd', 'linkage', 'eps', 'min_samples', 'min_cluster_size', 'n_clusters', 'noise_pct', 'silhouette', 'ari', 'nmi']]


Đã lưu thinh_so_sanh_thuat_toan.png (ghi nguồn vào figures/FIGURE_INDEX.csv)


,K,svd,linkage,eps,min_samples,min_cluster_size,n_clusters,noise_pct,silhouette,ari,nmi
algo,,,,,,,,,,,
hdbscan,NaN,50,NaN,NaN,NaN,100.0,8,0.505500,0.492474,0.194549,0.532533
dbscan,NaN,50,NaN,0.5,5.0,NaN,53,0.111833,0.077143,0.393648,0.554132
agglomerative,10.0,50,average,NaN,NaN,NaN,10,0.000000,0.257083,0.554307,0.653224
minibatch_kmeans,10.0,100,NaN,NaN,NaN,NaN,10,0.000000,0.179643,0.544952,0.705694
kmeans,10.0,50,NaN,NaN,NaN,NaN,10,0.000000,0.301682,0.622955,0.731825


> **Kết luận mục 3 (15/11):**
> - Thuật toán phân cụm tốt nhất: K-Means với SVD 50 (NMI đạt 0.732).
> - Vì sao DBSCAN/HDBSCAN kém trên dữ liệu này: Các ngành văn bản không tách biệt thành các vùng mật độ riêng biệt rõ ràng; độ mật độ không đồng đều giữa các nhóm ngành khiến việc gom cụm dựa trên mật độ bị đẩy phần lớn dữ liệu thành điểm nhiễu.
> - Bài học về Silhouette: Silhouette có thể bị đánh lừa lên rất cao (như DBSCAN đạt 0.678) khi thuật toán loại bỏ phần lớn dữ liệu làm nhiễu và chỉ giữ lại các cụm cực kỳ thưa thớt bên trong.
> - Ngày mai: Tiếp tục thực hiện t-SNE (mục 4) để trực quan hóa không gian phân cụm.
```

## 4. t-SNE

In [3]:
from sklearn.manifold import TSNE

Z50 = embed(50)
km = KMeans(10, n_init=10, random_state=42).fit(Z50)     # cấu hình phân cụm tốt nhất (NMI 0.7318)
idx = np.random.RandomState(0).choice(len(df), 6000, replace=False)
Zs, yy, cum = Z50[idx], y[idx], km.labels_[idx]

P = TSNE(2, perplexity=30, random_state=42, init='pca').fit_transform(Zs)   # khoảng 1 phút

fig, ax = plt.subplots(1, 2, figsize=(16, 6.5))
for c in np.unique(yy):
    ax[0].scatter(*P[yy == c].T, s=3, label=c)
for k in range(10):
    ax[1].scatter(*P[cum == k].T, s=3, label=f'cụm {k}')
ax[0].set_title('Tô màu theo ngành thật')
ax[1].set_title('Tô màu theo cụm K-Means (K = 10, SVD 50)')
for a in ax:
    a.legend(markerscale=4, fontsize=7, loc='best'); a.set_xticks([]); a.set_yticks([])
fig.suptitle('t-SNE trên 6.000 tiêu đề (không gian SVD 50)')
fig.tight_layout()
save_fig(fig, 'thinh_tsne_nganh_vs_cum', notebook='07_Thinh_PhanCum.ipynb', member='thinh', cell='Mục 4 - t-SNE',
         y_nghia='(tự viết)')


Đã lưu thinh_tsne_nganh_vs_cum.png (ghi nguồn vào figures/FIGURE_INDEX.csv)


WindowsPath('D:/mayhoc-git/DoAnCuoiKy/figures/thinh_tsne_nganh_vs_cum.png')

> **Nhận xét t-SNE (16/11):**
> - Những ngành nào tách thành đám riêng rõ ràng? Các ngành như CPUs, Digital Cameras, TVs, và Mobile Phones tạo thành các đám phân bố riêng biệt khá rõ ràng trên không gian giảm chiều nhờ từ vựng đặc trưng riêng
> - Những ngành nào nằm chồng lên nhau? Ba ngành thiết bị điện lạnh gia dụng gồm Fridges, Freezers và Fridge Freezers nằm chồng lấn, lẫn lộn vào nhau do có chung nhiều thuật ngữ mô tả
> - So sánh hai hình: Cụm K-Means ($K=10$) phân chia bám sát và khớp khá tốt với các đám màu của ngành thật ở hầu hết các danh mục, ngoại trừ nhóm thiết bị lạnh bị trộn lẫn
> - Lưu ý: t-SNE chỉ mang tính chất trực quan hóa cấu trúc láng giềng, khoảng cách hình học giữa các đám trên biểu đồ không phản ánh chính xác khoảng cách tuyệt đối trong không gian gốc

5 Cụm K-Means và ngành thật – chốt thuật toán

In [4]:
bang = pd.crosstab(pd.Series(km.labels_, name='cụm'), pd.Series(y, name='ngành'))
tom_tat = pd.DataFrame({'ngành chiếm đa số': bang.idxmax(axis=1),
                        'số mẫu': bang.sum(axis=1),
                        'độ thuần (%)': (bang.max(axis=1) / bang.sum(axis=1) * 100).round(1)})
print('Ngành không chiếm đa số ở cụm nào:', sorted(set(y) - set(tom_tat['ngành chiếm đa số'])))

ty_le = bang.div(bang.sum(axis=1), axis=0)          # tỉ lệ ngành trong mỗi cụm (mỗi hàng cộng = 100%)
fig, ax = plt.subplots(figsize=(10, 5.5))
im = ax.imshow(ty_le, cmap='Blues', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(len(bang.columns))); ax.set_xticklabels(bang.columns, rotation=40, ha='right', fontsize=8)
ax.set_yticks(range(len(bang.index)))
ax.set_yticklabels([f'cụm {k} ({n} mẫu)' for k, n in zip(bang.index, bang.sum(axis=1))], fontsize=8)
for i in range(ty_le.shape[0]):
    for j in range(ty_le.shape[1]):
        if ty_le.iloc[i, j] >= 0.05:
            ax.text(j, i, f'{ty_le.iloc[i, j]:.0%}', ha='center', va='center', fontsize=7,
                    color='white' if ty_le.iloc[i, j] > 0.5 else 'black')
fig.colorbar(im, ax=ax, fraction=0.03)
ax.set_title('Thành phần ngành trong từng cụm K-Means (K = 10, SVD 50)')
fig.tight_layout()
save_fig(fig, 'thinh_cum_vs_nganh', notebook='07_Thinh_PhanCum.ipynb', member='thinh', cell='Mục 5 - cụm vs ngành',
         y_nghia='(tự viết)')
tom_tat

Ngành không chiếm đa số ở cụm nào: ['Freezers']
Đã lưu thinh_cum_vs_nganh.png (ghi nguồn vào figures/FIGURE_INDEX.csv)


,ngành chiếm đa số,số mẫu,độ thuần (%)
cụm,,,
0,Mobile Phones,3696,98.5
1,Dishwashers,2350,95.2
2,Fridge Freezers,5030,83.2
3,TVs,3356,97.6
4,Washing Machines,2549,100.0
5,CPUs,3870,99.4
6,Microwaves,1932,98.9
7,Dishwashers,3604,27.6
8,Digital Cameras,2699,95.9


> **Kết luận phân cụm (16/11):**
> - Thuật toán chọn: K-Means với SVD 50 ($K=10$), đạt NMI cao nhất (0.732) và ARI (0.623), vượt trội hoàn toàn so với các thuật toán MiniBatch, Agglomerative, DBSCAN và HDBSCAN
> - Cụm nào khớp gần như hoàn toàn một ngành? Các cụm như Washing Machines (độ thuần 100%), CPUs (99.4%), Microwaves (98.9%), Mobile Phones (98.5%) khớp gần như tuyệt đối.
> - Cụm nào bị trộn nhiều ngành? Cụm 7 và Cụm 9 bị trộn lẫn giữa các dòng thiết bị điện lạnh (Fridges, Freezers, Fridge Freezers) do đặc điểm từ vựng mô tả sản phẩm rất tương đồng.
> - Vì sao Freezers không có cụm riêng? Do số lượng mẫu Freezers ít và có từ khóa gần giống Fridge Freezers nên bị gom chung vào cụm của Fridge Freezers.
> - Hạn chế và hướng phát triển: Mô hình phân cụm tốt ở các ngành hàng tiêu dùng điện tử lớn nhưng còn nhầm lẫn ở nhóm hàng gia dụng tương tự nhau; có thể cải thiện bằng cách bổ sung đặc trưng tên danh mục.